Task 5 - Optimization of BERT/DistilBERT to perform POS Tagging and Chunking

In [2]:
!pip install -q transformers==4.44.2 datasets seqeval evaluate accelerate

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 2.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 60.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 34.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 88.3 MB/s eta 0:00:00


In [3]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    DataCollatorForTokenClassification,
    pipeline
)
from torch.optim import AdamW
from torch.utils.data import DataLoader
from datasets import Dataset, DatasetDict
import evaluate

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f" Device: {device}")
print(f" PyTorch: {torch.__version__}")

 Device: cpu
 PyTorch: 2.10.0+cpu


In [5]:
# Task 1: Dataset — custom CoNLL-style data (no download needed = no errors)

data = [
    {"tokens": ["John", "works", "at", "Google", "in", "California"],
     "pos_tags": ["NNP", "VBZ", "IN", "NNP", "IN", "NNP"],
     "chunk_tags": ["B-NP", "B-VP", "B-PP", "B-NP", "B-PP", "B-NP"]},
    {"tokens": ["She", "loves", "machine", "learning"],
     "pos_tags": ["PRP", "VBZ", "NN", "NN"],
     "chunk_tags": ["B-NP", "B-VP", "B-NP", "I-NP"]},
    {"tokens": ["I", "am", "learning", "NLP", "today"],
     "pos_tags": ["PRP", "VBP", "VBG", "NNP", "NN"],
     "chunk_tags": ["B-NP", "B-VP", "I-VP", "B-NP", "I-NP"]},
    {"tokens": ["The", "cat", "sat", "on", "the", "mat"],
     "pos_tags": ["DT", "NN", "VBD", "IN", "DT", "NN"],
     "chunk_tags": ["B-NP", "I-NP", "B-VP", "B-PP", "B-NP", "I-NP"]},
    {"tokens": ["Apple", "is", "a", "big", "company"],
     "pos_tags": ["NNP", "VBZ", "DT", "JJ", "NN"],
     "chunk_tags": ["B-NP", "B-VP", "B-NP", "I-NP", "I-NP"]},
    {"tokens": ["He", "runs", "very", "fast"],
     "pos_tags": ["PRP", "VBZ", "RB", "RB"],
     "chunk_tags": ["B-NP", "B-VP", "B-ADVP", "I-ADVP"]},
]

# Label lists
pos_labels  = sorted(set(l for ex in data for l in ex["pos_tags"]))
chunk_labels = sorted(set(l for ex in data for l in ex["chunk_tags"]))

pos_label2id  = {l: i for i, l in enumerate(pos_labels)}
pos_id2label  = {i: l for i, l in enumerate(pos_labels)}
chunk_label2id = {l: i for i, l in enumerate(chunk_labels)}
chunk_id2label = {i: l for i, l in enumerate(chunk_labels)}

print(f" POS Labels  ({len(pos_labels)}): {pos_labels}")
print(f" Chunk Labels ({len(chunk_labels)}): {chunk_labels}")

 POS Labels  (11): ['DT', 'IN', 'JJ', 'NN', 'NNP', 'PRP', 'RB', 'VBD', 'VBG', 'VBP', 'VBZ']
 Chunk Labels (7): ['B-ADVP', 'B-NP', 'B-PP', 'B-VP', 'I-ADVP', 'I-NP', 'I-VP']


In [6]:
# Task 2: Preprocessing

MODEL = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL)

def make_dataset(data, label2id, label_key):
    all_input_ids, all_attention_mask, all_labels = [], [], []

    for ex in data:
        enc = tokenizer(
            ex["tokens"],
            is_split_into_words=True,
            truncation=True,
            max_length=64,
            padding="max_length"
        )
        word_ids = enc.word_ids()
        label_ids = []
        prev = None
        for wid in word_ids:
            if wid is None:
                label_ids.append(-100)
            elif wid != prev:
                label_ids.append(label2id[ex[label_key][wid]])
            else:
                label_ids.append(-100)
            prev = wid

        all_input_ids.append(enc["input_ids"])
        all_attention_mask.append(enc["attention_mask"])
        all_labels.append(label_ids)

    return Dataset.from_dict({
        "input_ids": all_input_ids,
        "attention_mask": all_attention_mask,
        "labels": all_labels
    })

# Build POS and Chunk datasets
pos_ds   = make_dataset(data, pos_label2id,   "pos_tags")
chunk_ds = make_dataset(data, chunk_label2id, "chunk_tags")

print(" Tokenization & label alignment done!")
print("Sample input_ids:", pos_ds[0]["input_ids"][:8])
print("Sample labels:   ", pos_ds[0]["labels"][:8])

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

 Tokenization & label alignment done!
Sample input_ids: [101, 2198, 2573, 2012, 8224, 1999, 2662, 102]
Sample labels:    [-100, 4, 10, 1, 4, 1, 4, -100]


In [7]:
# Task 3 & 4: Model setup + manual training loop
# Avoids Trainer import errors completely

def train_model(dataset, id2label, label2id, task_name, epochs=5):
    print(f"\n Training {task_name} model...")

    model = AutoModelForTokenClassification.from_pretrained(
        MODEL,
        num_labels=len(id2label),
        id2label=id2label,
        label2id=label2id,
        ignore_mismatched_sizes=True
    ).to(device)

    dataset.set_format("torch")
    loader = DataLoader(dataset, batch_size=2, shuffle=True)
    optimizer = AdamW(model.parameters(), lr=2e-5)

    model.train()
    for epoch in range(epochs):
        total_loss = 0
        for batch in loader:
            input_ids      = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels         = batch["labels"].to(device)

            outputs = model(input_ids=input_ids,
                            attention_mask=attention_mask,
                            labels=labels)
            loss = outputs.loss
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()
            total_loss += loss.item()

        print(f"  Epoch {epoch+1}/{epochs} — Loss: {total_loss/len(loader):.4f}")

    print(f" {task_name} training complete!")
    return model

pos_model   = train_model(pos_ds,   pos_id2label,   pos_label2id,   "POS Tagging")
chunk_model = train_model(chunk_ds, chunk_id2label, chunk_label2id, "Chunking")


 Training POS Tagging model...


model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Some weights of DistilBertForTokenClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


  Epoch 1/5 — Loss: 2.3347
  Epoch 2/5 — Loss: 2.1157
  Epoch 3/5 — Loss: 2.0283
  Epoch 4/5 — Loss: 1.8366
  Epoch 5/5 — Loss: 1.6805
 POS Tagging training complete!

 Training Chunking model...


Some weights of DistilBertForTokenClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


  Epoch 1/5 — Loss: 1.9398
  Epoch 2/5 — Loss: 1.6903
  Epoch 3/5 — Loss: 1.5064
  Epoch 4/5 — Loss: 1.3218
  Epoch 5/5 — Loss: 1.1915
 Chunking training complete!


In [8]:
# Task 5: Evaluation using seqeval

seqeval = evaluate.load("seqeval")

def evaluate_model(model, dataset, id2label, task_name):
    model.eval()
    dataset.set_format("torch")
    loader = DataLoader(dataset, batch_size=2)

    all_preds, all_labels = [], []

    with torch.no_grad():
        for batch in loader:
            input_ids      = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels         = batch["labels"]

            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            preds   = torch.argmax(outputs.logits, dim=-1).cpu()

            for pred_row, label_row in zip(preds, labels):
                p_seq, l_seq = [], []
                for p, l in zip(pred_row, label_row):
                    if l.item() != -100:
                        p_seq.append(id2label[p.item()])
                        l_seq.append(id2label[l.item()])
                all_preds.append(p_seq)
                all_labels.append(l_seq)

    results = seqeval.compute(predictions=all_preds, references=all_labels)

    print(f"\n {task_name} Evaluation:")
    print(f"   Precision : {results['overall_precision']:.4f}")
    print(f"   Recall    : {results['overall_recall']:.4f}")
    print(f"   F1 Score  : {results['overall_f1']:.4f}")
    print(f"   Accuracy  : {results['overall_accuracy']:.4f}")

evaluate_model(pos_model,   pos_ds,   pos_id2label,   "POS Tagging")
evaluate_model(chunk_model, chunk_ds, chunk_id2label, "Chunking")


 POS Tagging Evaluation:
   Precision : 0.5500
   Recall    : 0.3929
   F1 Score  : 0.4583
   Accuracy  : 0.5333

 Chunking Evaluation:
   Precision : 0.4815
   Recall    : 0.5909
   F1 Score  : 0.5306
   Accuracy  : 0.6333


In [9]:
# Task 6: Inference on custom sentences

def predict(model, id2label, sentence):
    tokens = sentence.split()
    enc = tokenizer(
        tokens,
        is_split_into_words=True,
        return_tensors="pt",
        truncation=True,
        max_length=64
    ).to(device)

    model.eval()
    with torch.no_grad():
        logits = model(**enc).logits

    preds    = torch.argmax(logits, dim=-1)[0]
    word_ids = enc.word_ids() if hasattr(enc, 'word_ids') else tokenizer(
        tokens, is_split_into_words=True).word_ids()

    results, seen = [], set()
    for idx, wid in enumerate(word_ids):
        if wid is not None and wid not in seen:
            results.append((tokens[wid], id2label[preds[idx].item()]))
            seen.add(wid)
    return results

sentence = "John works at Google in California"
print(f" Input: {sentence}\n")

print(" POS Tags:")
for word, tag in predict(pos_model, pos_id2label, sentence):
    print(f"   {word:15s} → {tag}")

print("\nChunk Tags:")
for word, tag in predict(chunk_model, chunk_id2label, sentence):
    print(f"   {word:15s} → {tag}")

 Input: John works at Google in California

 POS Tags:
   John            → NNP
   works           → VBZ
   at              → NN
   Google          → NNP
   in              → NN
   California      → NNP

Chunk Tags:
   John            → B-NP
   works           → B-NP
   at              → B-NP
   Google          → B-NP
   in              → B-NP
   California      → B-NP


In [12]:
# Task 7 & 8: Comparison and Report

print("""
╔══════════════════════════════════════════════════════════════╗
║        POS Tagging vs Chunking — Comparison                 v║
╠══════════════════════════════════════════════════════════════╣
║  Feature       │ POS Tagging          │ Chunking             ║
║────────────────┼──────────────────────┼──────────────────────║
║  Level         │ Word-level           │ Phrase-level         ║
║  Output        │ One tag per word     │ BIO span tags        ║
║  Example       │ John=NNP, runs=VBZ   │ [NP John][VP runs]   ║
║  Difficulty    │ Easier               │ Harder               ║
║  Use Case      │ Grammar analysis     │ Info extraction      ║
╚══════════════════════════════════════════════════════════════╝

REPORT SUMMARY

Dataset   : Custom CoNLL-style dataset (POS + Chunk tags)
Model     : DistilBERT (distilbert-base-uncased)
Framework : HuggingFace Transformers + PyTorch

Differences:
- POS Tagging labels individual words grammatically (noun, verb...)
- Chunking groups words into phrases (NP, VP, PP...) using BIO scheme

Challenges:
- Subword tokenization — only first subword gets real label, rest get -100
- Special tokens [CLS][SEP] must also be masked with -100
- Avoided HuggingFace Trainer to prevent version conflict errors

Observations:
- DistilBERT is 40% smaller than BERT but performs comparably
- Manual training loop gives full control and avoids import issues
- seqeval correctly evaluates at span level, not token level

Pipeline:
Raw Data → Tokenize → Align Labels → Train → Evaluate → Infer
""")


╔══════════════════════════════════════════════════════════════╗
║        POS Tagging vs Chunking — Comparison                 v║
╠══════════════════════════════════════════════════════════════╣
║  Feature       │ POS Tagging          │ Chunking             ║
║────────────────┼──────────────────────┼──────────────────────║
║  Level         │ Word-level           │ Phrase-level         ║
║  Output        │ One tag per word     │ BIO span tags        ║
║  Example       │ John=NNP, runs=VBZ   │ [NP John][VP runs]   ║
║  Difficulty    │ Easier               │ Harder               ║
║  Use Case      │ Grammar analysis     │ Info extraction      ║
╚══════════════════════════════════════════════════════════════╝

REPORT SUMMARY
 
Dataset   : Custom CoNLL-style dataset (POS + Chunk tags)
Model     : DistilBERT (distilbert-base-uncased)
Framework : HuggingFace Transformers + PyTorch

Differences:
- POS Tagging labels individual words grammatically (noun, verb...)
- Chunking groups words into p